In [18]:
import json 
import pandas as pd
import bokeh 
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, HoverTool, CustomJSTickFormatter, ColorBar, Legend, LegendItem
import numpy as np
from bokeh.transform import linear_cmap
from bokeh.plotting import output_notebook

json_path = 'chatbox_context_token_date.json'
json_file = json.load(open(json_path))

df = pd.DataFrame(json_file).T

def convert_size(size_str):
    if size_str.endswith('K'):
        return int(float(size_str[:-1]) * 1e3)
    elif size_str.endswith('M'):
        return int(float(size_str[:-1]) * 1e6)
    elif size_str.endswith('B'):
        return int(float(size_str[:-1]) * 1e9)
    elif size_str.endswith('T'):
        return int(float(size_str[:-1]) * 1e12)
    elif size_str == "None":
        return -1
    else:
        return int(size_str)

def get_size(parameters):
    if parameters is None:
        return 3
    elif parameters == -1:
        return 3
    else:
        return (np.log10(parameters)-7)*3

palette = list(reversed(bokeh.palettes.YlOrRd7))
df['int_parameters'] = df['expected_parameters'].apply(lambda x: convert_size(x))
df['int_context_length'] = df['context_length'].apply(lambda x: convert_size(x))
df['int_max_tokens'] = df['max_tokens'].apply(lambda x: convert_size(x))
df['log_max_tokens'] = df['int_max_tokens'].apply(lambda x: np.log10(x) if (x is not None and x != -1) else -1)
df['released'] = pd.to_datetime(df['released'])
df['size'] = df['int_parameters'].apply(get_size)

source = ColumnDataSource(df)

p = figure(x_axis_type='datetime', 
           title="Evolution taille de context des LLMs", 
           width=800, 
           height=400)

mapper = linear_cmap('int_max_tokens',
                     palette, 
                     df['int_max_tokens'].min(), 
                     df['int_max_tokens'].max())

p.scatter(x='released', 
          y='int_context_length', 
          size='size',
          source=source, 
          fill_color=mapper,
          line_color='#292828', alpha=0.4)

p.yaxis.formatter = CustomJSTickFormatter(code="""
    if (tick >= 1e6) {
        return (tick / 1e6) + 'M';
    } else if (tick >= 1e3) {
        let val = tick / 1e3;
        return val + (val === 100 ? 'k' : 'K');
    } else {
        return tick;
    }
""")

color_bar = ColorBar(color_mapper=mapper['transform'], 
                     orientation='horizontal',
                     title="Max Output Tokens")
p.add_layout(color_bar, 'below')

# legend_items = []
# for val, label in [(1e7, '10M'), (1e9, '1B'), (1e11, '100B')]:
#     dummy_source = ColumnDataSource(dict(released=[pd.NaT], int_context_length=[np.nan]))
#     dummy = p.scatter(x='released', y='int_context_length', size=get_size(val), 
#                       source=dummy_source, line_color='#292828', fill_color='white', alpha=0.4)
#     legend_items.append(LegendItem(label=label, renderers=[dummy]))

# size_legend = Legend(items=legend_items, orientation="horizontal", title="Taille (Paramètres)")
# p.add_layout(size_legend, 'below')

hover = HoverTool(tooltips=[
    ("Model", "@index"),
    ("Parameters", "@parameters"),
    ("Released", "@released{%F}"),
    ("Knowledge Cutoff", "@knowledge_cutoff"),
    ("Context Length", "@context_length"),
    ("Output token", "@max_tokens"),
], formatters={'@released': 'datetime'})

p.add_tools(hover)
output_notebook()
show(p)

display(df)

Loading BokehJS ...

,context_length,max_tokens,knowledge_cutoff,released,expected_parameters,parameters,particularity,difference_from_previous,int_parameters,int_context_length,int_max_tokens,log_max_tokens,size
GPT-1,None,None,2018-10-01,2018-06-11,117M,117M,NaN,NaN,117000000,-1,-1,-1.00000,3.204558
GPT-2,None,None,2017-12-01,2019-02-14,1.5B,1.5B,NaN,NaN,1500000000,-1,-1,-1.00000,6.528274
GPT-3,None,None,2020-10-01,2020-06-11,175B,175B,NaN,NaN,175000000000,-1,-1,-1.00000,12.729114
GPT-3.5,None,None,2021-09-01,2022-11-01,175B,None,NaN,NaN,175000000000,-1,-1,-1.00000,12.729114
GPT-3.5-turbo,16385,4096,2021-07-01,2023-05-28,175B,175B,NaN,NaN,175000000000,16385,4096,3.61236,12.729114
GPT-3.5-turbo-instruct,4096,4096,2021-07-01,2023-07-28,175B,175B,NaN,NaN,175000000000,4096,4096,3.61236,12.729114
GPT-3.5-turbo-1106,16385,4096,2021-07-01,2023-11-05,175B,175B,NaN,NaN,175000000000,16385,4096,3.61236,12.729114
GPT-4,8192,8192,2023-12-01,2023-06-01,1.75T,1.75T,8 Mixture of experts of 220B,multimodality text + image,1750000000000,8192,8192,3.91339,15.729114
GPT-4o,128K,4096,2023-10-01,2024-05-12,200B,200B,NaN,NaN,200000000000,128000,4096,3.61236,12.903090
GPT-4o mini,128K,16384,2023-10-01,2024-07-17,8B,8B,NaN,NaN,8000000000,128000,16384,4.21442,8.709270
